In [ ]:
from common import *

# most: 07a (imputacija potpuno nedostajucih atributa, sekcija 11.1) je
# zavrsena i sacuvala 'weatherAusAfter11_1_4.csv' - nastavljamo od tog stanja
data = loadData("backups/weatherAusAfter11_1_4.csv")


### 11.2 Nasumicno nedostajuce vrednosti

#### 11.2.1 Analiza raspodele

Analizirani dataset karakteriše značajan broj nedostajućih podataka. Prikazimo broj nedostajucih vrednosti za svaku kolonu.
Sortiranje je izvršeno u rastućem redosledu, polazeći od pretpostavke da će broj nedostajućih podataka po koloni opredeliti redosled kasnije imputacije.

In [ ]:
data = loadData("backups/weatherAusAfter11_1_4.csv")
originalDF = loadData("InputData/weatherAUS.csv")

originalCols = [x for x in data.columns if x in originalDF.columns]

print(originalCols)
print(data.isna().sum().sort_values(ascending=True))

Pored nedostajucih podataka po kolonama, pogledajmo i broj nedostajucih podataka po vrstama.

In [ ]:
missing_counts = data[originalCols].isna().sum(axis=1)

counts = missing_counts.value_counts().sort_index()

print(counts)

Primecujemo da postoji odredjeni broj vrsta u kojima fali i do 19 podataka. Ako uzmemo u obzir da dataset ukupno ima 24 kolone i da su od toga barem tri kolone neophodne za rad (bez nedostajucih vrednosti), dolazimo do zaključka da one vrste u kojima nam fali veliki broj podataka treba obrisati.

In [ ]:
POTREBAN_BROJ_PODATAKA_U_REDU = 15
data = data.dropna(subset=originalCols,thresh=POTREBAN_BROJ_PODATAKA_U_REDU)
print(data.isna().sum().sort_values(ascending=True))


write_log(data, "Brisanje vrsta sa velikim brojem nedostajucih vrednosti", "WatherAus_After_11_2_1.csv")


#### 11.2.2 Metode imputacije

U nastavku se nalaze dve metode za imputaciju nasumično nedostajućih vrednosti (za razliku od odeljka 11.1, gde su vrednosti u potpunosti nedostajale na nivou cele stanice): Random Forest regresija, koja za predviđanje nedostajuće vrednosti koristi meteorološki i geografski povezane prediktore iz istog reda, i ponderisani prosek po inverznoj udaljenosti (IDW), koji se oslanja isključivo na istovremena očitavanja sa obližnjih stanica.

Izbor metode po atributu zavisi od prirode fizičke veličine: IDW je pogodniji za atribute koji se prostorno menjaju postepeno i bez naglih lokalnih oscilacija (npr. atmosferski pritisak - videćemo ga u primeni kod `Pressure9am`/`Pressure3pm`), dok se Random Forest koristi kada je zavisnost od ostalih merenja na istoj stanici jača od prostorne zavisnosti, ili kada za dati atribut ne postoji dovoljno gusta mreža obližnjih stanica sa istovremenim očitavanjima.

##### 11.2.2.1 Random forest

In [ ]:

core_features = ["Nadmorska visina (m)","Inv_Dist_Sever","Inv_Dist_Jug","Inv_Dist_Istok","Inv_Dist_Zapad",
"klima_Grassland","klima_Subtropical","klima_Temperate","klima_Tropical"]
def imputeData(target_attr, df, prediktori, stvarnaImputacija = False, backupName=None):
    df = df.loc[:, ~df.columns.duplicated()]  # odbrana od dupliranih kolona (npr. Inv_Dist_Jug) koje mogu nastati u prethodnim mergovima
    broj_nedostajucih = df[target_attr].isna().sum()
    if broj_nedostajucih == 0:
        print(f"Kolona '{target_attr}' nema nedostajućih vrednosti. Imputacija nije potrebna.")
        return df
    proc_nedostajucih = (df[target_attr].isna().sum() / len(df)) * 100
    global core_features

    # deduplikacija - core_features se ponekad poklapa sa nekim od eksplicitno
    # prosledjenih prediktora (npr. Inv_Dist_Jug za Humidity3pm), sto bi bez ovoga
    # dalo dupliranu kolonu u poznati_df[prediktori] i puklo pri fit-u imputer-a
    prediktori = list(dict.fromkeys(prediktori + core_features))
    poznati_df = df[df[target_attr].notna() & (df['Date'] < SPLIT_DATE)].copy()
    nedostajuci_df = df[df[target_attr].isna()].copy()

   
    imputer = SimpleImputer(strategy='median')
    X_poznati = imputer.fit_transform(poznati_df[prediktori])
    y_poznati = poznati_df[target_attr].values

    metrics = {'mae': [], 'rmse': [], 'r2': [], 'adj_r2': []}

    # evaluacija je samo za MAE/RMSE/R2 u logu/izvestaju - ne utice na finalnu
    # imputaciju (rf_final ispod se trenira nezavisno, sa n_estimators=100 kao pre).
    # Skraceno sa 5 na 2 folda i manje stabala da evaluacija ne bude usko grlo.
    for i in range(2):
        X_train, X_test, y_train, y_test = train_test_split(
            X_poznati, y_poznati, test_size=0.1, random_state=i
        )
        
        rf = RandomForestRegressor(n_estimators=30, random_state=i, n_jobs=-1)
        rf.fit(X_train, y_train)
        
        y_pred = rf.predict(X_test)
        
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)
        
        n = len(y_test)
        p = X_test.shape[1]
        adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
        
        metrics['mae'].append(mae)
        metrics['rmse'].append(rmse)
        metrics['r2'].append(r2)
        metrics['adj_r2'].append(adj_r2)

    avg_mae = np.mean(metrics['mae'])
    avg_rmse = np.mean(metrics['rmse'])
    avg_r2 = np.mean(metrics['r2'])
    avg_adj_r2 = np.mean(metrics['adj_r2'])

    spisak_ocena = f"MAE={avg_mae:.4f}, RMSE={avg_rmse:.4f}, R2={avg_r2:.4f}, Adj R2={avg_adj_r2:.4f}, postoji {proc_nedostajucih:.2f}% nedostjucih podaataka u koloni"

    poruka = f"imputirani nedostajući podaci za \natribut {target_attr}, \nmetoda RF, \nprediktori: {', '.join(prediktori)}, \nstatističke ocene: {spisak_ocena}"

    print("--- Evaluacija završena ---")
    print(poruka)
    print("---------------------------\n")

    if stvarnaImputacija:
        rf_final = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        rf_final.fit(X_poznati, y_poznati)

        X_nedostajuci = imputer.transform(nedostajuci_df[prediktori])

        imputirane_vrednosti = rf_final.predict(X_nedostajuci)

        noviDF = df.copy()
        noviDF.loc[noviDF[target_attr].isna(), target_attr] = imputirane_vrednosti
        proc_nedostajucih_sad = (noviDF[target_attr].isna().sum() / len(noviDF)) * 100
        print(f"Stvarna imputacija zavrsena, u koloni sada ima {proc_nedostajucih_sad}% nedostajucih podataka")
        write_log(noviDF, poruka, backupName)
        noviDF = createVaribales(noviDF)
        return noviDF
    return df

##### 11.2.2.2 Ponderisani prosek (inverzna udaljenost)
Naredni kod koristi metodu ponderisanog proseka, odnosno inverznu udaljenost, za popunjavanje nedostajućih vrednosti. Ovaj prostorni pristup se oslanja na podatke sa susednih mernih stanica, dajući veću težinu onim stanicama koje su fizički bliže. Ovo je dobro rešenje za imputiranje nedostajućih vrednosti za atmosferski pritisak jer je to meteorološka varijabla koja se menja postepeno na širem geografskom području i ne pokazuje nagle lokalne oscilacije. Zbog prirodne prostorne korelacije, očitavanja sa obližnjih stanica su izuzetno pouzdani indikatori stanja na ciljnoj lokaciji. U slučajevima gde postoje adekvatni podaci u okolini, ova metoda ostvaruje visoke rezultate i veoma precizno preslikava stvarno stanje na terenu.

In [ ]:
def imputeData_IDW(target_attr, df, df_dist, max_dist, stvarnaImputacija=False, backupName=None):
    proc_nedostajucih = df[target_attr].isna().sum()
    
    print("Pravljenje rečnika prostornih udaljenosti...")
    dist_dict = {}
    for _, row in df_dist.iterrows():
        s1, s2, d = row['stanica_1'], row['stanica_2'], row['razdaljina_km']
        if d <= max_dist:
            dist_dict.setdefault(s1, {})[s2] = d
            dist_dict.setdefault(s2, {})[s1] = d

    def dobiji_susede(stanica, datum, lookup_dict):
        susedi_info = dist_dict.get(stanica, {})
        validni_susedi = []
        for sused, dist in susedi_info.items():
           
            vrednost_suseda = lookup_dict.get((sused, datum))
            if vrednost_suseda is not None and pd.notnull(vrednost_suseda):
                validni_susedi.append({
                    'value': vrednost_suseda,
                    'distance': dist
                })
        return validni_susedi

    poznati_indeksi = df[df[target_attr].notna()].index
    nedostajuci_indeksi = df[df[target_attr].isna()].index

    metrics = {'mae': [], 'rmse': [], 'r2': [], 'adj_r2': []}

    print("Započinjem evaluaciju (5 iteracija)...")
    # evaluacija je samo za MAE/RMSE/R2 u logu/izvestaju - ne utice na finalnu
    # imputaciju ispod. Skraceno sa 5 na 2 folda da evaluacija ne bude usko grlo.
    for i in range(2):
        np.random.seed(i)
        
        test_size = int(0.1 * len(poznati_indeksi))
        test_indeksi = np.random.choice(poznati_indeksi, size=test_size, replace=False)
        train_indeksi = poznati_indeksi.difference(test_indeksi)

        df_train = df.loc[train_indeksi]
        train_lookup = df_train.set_index(['Location', 'Date'])[target_attr].to_dict()
        y_test = []
        y_pred = []

        for idx in test_indeksi:
            red = df.loc[idx]
            susedi = dobiji_susede(red['Location'], red['Date'], train_lookup)
            
            if not susedi:
                continue
            
            vrednosti = [s['value'] for s in susedi]
            udaljenosti = [s['distance'] for s in susedi]
            tezine = [1.0 / (d + 1e-6) for d in udaljenosti]
            
            y_pred.append(np.average(vrednosti, weights=tezine))
            y_test.append(red[target_attr])

        if len(y_test) > 0:
            mae = mean_absolute_error(y_test, y_pred)
            rmse = np.sqrt(mean_squared_error(y_test, y_pred))
            r2 = r2_score(y_test, y_pred)
            
            n = len(y_test)
            p = 1 
            adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1) if n > p + 1 else np.nan
            
            metrics['mae'].append(mae)
            metrics['rmse'].append(rmse)
            metrics['r2'].append(r2)
            metrics['adj_r2'].append(adj_r2)
    ukupno_nedostajucih = len(nedostajuci_indeksi)
    broj_popunjivih = 0
    

    full_lookup = df.set_index(['Location', 'Date'])[target_attr].to_dict()
    
    if ukupno_nedostajucih > 0:
        for idx in nedostajuci_indeksi:
            red = df.loc[idx]
            if dobiji_susede(red['Location'], red['Date'], full_lookup):
                broj_popunjivih += 1
        proc_popunjivih = (broj_popunjivih / ukupno_nedostajucih) * 100
    else:
        proc_popunjivih = 0.0

    if metrics['mae']:
        avg_mae = np.mean(metrics['mae'])
        avg_rmse = np.mean(metrics['rmse'])
        avg_r2 = np.mean(metrics['r2'])
        avg_adj_r2 = np.mean(metrics['adj_r2'])
        spisak_ocena = (f"MAE={avg_mae:.4f}, RMSE={avg_rmse:.4f}, R2={avg_r2:.4f}, Adj R2={avg_adj_r2:.4f}, "
                        f"početno nedostajućih={proc_nedostajucih:.2f} od cega je moguće popuniti={proc_popunjivih:.2f}%")
    else:
        spisak_ocena = "Evaluacija nemoguća (nema dovoljno validnih suseda u blizini)"

    poruka = f"Imputirani nedostajući podaci \natribut: {target_attr}), \nmetoda: Ponderisani prosek (Inverzna udaljenost, max_dist={max_dist}), \nocene: {spisak_ocena}"

    print("--- Evaluacija završena ---")
    print(poruka)
    print("---------------------------\n")

    if stvarnaImputacija:
        noviDF = df.copy()
        print("Započinjem stvarnu imputaciju podataka...")

        for idx in nedostajuci_indeksi:
            red = df.loc[idx]
            susedi = dobiji_susede(red['Location'], red['Date'], full_lookup)
            
            if susedi:
                vrednosti = [s['value'] for s in susedi]
                udaljenosti = [s['distance'] for s in susedi]
                tezine = [1.0 / (d + 1e-6) for d in udaljenosti]
                noviDF.at[idx, target_attr] = np.average(vrednosti, weights=tezine)
        write_log(noviDF, poruka, backupName)
        noviDF = createVaribales(noviDF)
        return noviDF

    return df

razdaljine_df = pd.read_csv("GeoPodaci/udaljenost_stanica.csv")


#### 11.2.3 Imputacija nedostajucih podataka

U nastavku će biti sprovedena sukcesivna imputacija nedostajućih vrednosti za sledeći skup atributa: MinTemp, MaxTemp, Temp9am, Temp3pm, Pressure9am, Pressure3pm, Humidity9am i Sunshine.

Redosled kojim se imputacija vrši strogo je logičan i uslovljen prirodom meteoroloških podataka, odnosno njihovom hronološkom i uzročno-posledičnom zavisnošću.

Za popunjavanje vrednosti svakog pojedinačnog atributa pozivaće se prethodno definisane funkcije za imputaciju. Pri svakom pozivu, ovim funkcijama će se prosleđivati isključivo specifični prediktori koji su najrelevantniji za taj konkretan atribut. Sa druge strane, univerzalni prediktori – koji se odnose na geografske i klimatske odlike pojedinačne merne stanice – već su ugrađeni u samu logiku funkcije za imputaciju, čime je obezbeđena konzistentnost i izbegnuto redundatno prosleđivanje argumenata.

##### 11.2.3.1 MinTemp


In [ ]:
data = loadData("backups/WatherAus_After_11_2_1.csv")
data = imputeData("MinTemp",data, ['Temp3pm_rolling_mean_3','Pressure9am', 'Temp9am', 
                                   'Temp3pm', 'Year_Scaled', 'DayOfYear_Sin', 'DayOfYear_Cos',
                                     'WindSpeed9am', 'Humidity9am'], True, "WeatherAus_After_11_2_1_1.csv")


##### 11.2.3.2 MaxTemp

In [ ]:
data = imputeData("MaxTemp",data, 
                  ['Temp3pm_rolling_mean_3','Pressure3pm', 'Temp9am', 
                   'Temp3pm', 'Year_Scaled', 'DayOfYear_Sin', 
                   'DayOfYear_Cos', 'WindSpeed3pm', 'Humidity3pm'], True,
                   "WeatherAus_After_11_2_1_2.csv")

##### 11.2.3.3 Temp9am

In [ ]:
data = imputeData("Temp9am",data, 
                  ['Temp3pm_rolling_mean_3','Pressure3pm', 'MinTemp', 
                   'Temp3pm', 'Year_Scaled', 'DayOfYear_Sin', 'DayOfYear_Cos', 
                   'WindSpeed3pm', 'Humidity3pm'], True,
                   "WeatherAus_After_11_2_1_3.csv")

##### 11.2.3.4 Temp3pm

In [ ]:
data = imputeData("Temp3pm",data, 
                  ['Temp3pm_rolling_mean_3','Pressure3pm', 'MinTemp', 
                   'Temp9am', 'Year_Scaled', 'DayOfYear_Sin', 'DayOfYear_Cos', 
                   'WindSpeed3pm', 'Humidity3pm'], True,
                   "WeatherAus_After_11_2_1_4.csv")


##### 11.2.3.5 Pressure9am

a) Ponderisani prosek 

In [ ]:
data = loadData("backups/WeatherAus_After_11_2_1_4.csv")
data = imputeData_IDW("Pressure9am", data, razdaljine_df, 150, True,
                    "WeatherAus_After_11_2_1_5a.csv")

b) Random forest 

Iako upotreba ponderisnog proseka daje odlične rezultate, ova metoda ne može da popuni apsolutno sve nedostajuće podatke. Zbog strogog oslanjanja na prostornu blizinu, algoritam neće moći da izvrši imputaciju u situacijama kada u definisanom maksimalnom radijusu nema drugih mernih stanica, ili kada susedne stanice takođe nemaju zabeležena očitavanja za taj konkretan dan. Kako bismo osigurali potpuno popunjen skup podataka, preostale vrednosti koje su ostale neimputirane biće rešene primenom Random Forest modela, po potpuno istom principu kao što je to rađeno i u ostalim slučajevima. 

In [ ]:
data = imputeData("Pressure9am",data, 
                          ["Pressure3pm",'Temp9am', 'Temp3pm', 'Year_Scaled', 
                           'DayOfYear_Sin', 'DayOfYear_Cos', 'Pressure9am_pre_1_dana', 
                           'Pressure3pm_pre_1_dana', 'Temp9am_Min_diff', 'Max_Temp3pm_diff', 
                           'WindSpeed3pm', 'WindDir3pm_sin', 'WindDir3pm_cos', 
                           'WindDir9am_sin', 'WindDir9am_cos', 'WindSpeed3pm_rolling_mean_3'], 
                           True, "WeatherAus_After_11_2_1_5b.csv")

##### 11.2.3.6 Pressure3pm

a) Ponderisani prosek

In [ ]:
data = imputeData_IDW("Pressure3pm", data, razdaljine_df, 150, 
                      True, "WeatherAus_After_11_2_1_6a.csv")

b) Random forest

In [ ]:
data = imputeData("Pressure3pm",data, 
                          ['Pressure9am', 'Temp9am', 'Temp3pm', 'Year_Scaled', 
                           'DayOfYear_Sin', 'DayOfYear_Cos', 'Pressure9am_pre_1_dana', 
                           'Pressure3pm_pre_1_dana', 'Temp9am_Min_diff', 
                           'Max_Temp3pm_diff', 'WindSpeed3pm', 'WindDir3pm_sin', 
                           'WindDir3pm_cos', 'WindDir9am_sin', 'WindDir9am_cos', 
                           'WindSpeed3pm_rolling_mean_3'], True, 
                           "WeatherAus_After_11_2_1_6b.csv")


##### 11.2.3.7 Humidity9am

Značaj tacke rose leži u njegovoj direktnoj matematičkoj vezi sa temperaturom i relativnom vlažnošću. Za razliku od relativne vlažnosti koja oscilira u zavisnosti od trenutne temperature, tačka rose je egzaktniji pokazatelj stvarne količine vlage u vazduhu. Fizička zakonitost je sledeća: što je razlika između trenutne temperature vazduha i tačke rose manja, to je relativna vlažnost veća. Zahvaljujući ovome, ukoliko su nam poznate trenutna temperatura i tačka rose, relativnu vlažnost možemo veoma precizno izračunati korišćenjem Magnus-Tetensove formule. Formula se oslanja na izračunavanje stvarnog pritiska vodene pare i pritiska zasićene pare, a vlažnost se dobija na sledeći način:
$$RH = 100 \times \frac{\exp\left(\frac{17.625 \cdot T_d}{243.04 + T_d}\right)}{\exp\left(\frac{17.625 \cdot T}{243.04 + T}\right)}$$

gde je:

$RH$ - relativna vlažnost vazduha (u procentima)

$T$ - trenutna temperatura vazduha (u °C)

$T_d$ - tačka rose (u °C)

Kako bismo eksperimentalno potvrdili da je tačka rose adekvatan posrednik za imputaciju vlažnosti, u kodu koji je dat u Prilogu primenjujemo navedenu formulu na onaj deo skupa podataka u kom su nam već poznate i temperatura i vlažnost vazduha (za 9 ujutru i 3 popodne). Kod matematičkim putem izračunava vlažnost, a zatim je upoređuje sa stvarnim, zabeleženim vrednostima senzora. Računanjem srednje apsolutne greške (MAE) i Pirsonove korelacije direktno ćemo proveriti preciznost i opravdanost ovog pristupa.

Kao što se i očekivalo na osnovu termodinamičkih zakonitosti, rezultati testiranja su sa visokim stepenom značajnosti potvrdili snažnu vezu između ovih veličina. Ovaj matematički pristup pokazao se mnogo boljim i pouzdanijim u odnosu na klasične prediktivne modele mašinskog učenja koje smo ranije razmatrali.

Sada prelazimo na primenu ove formule na celokupan dataset. Kod koji sledi vrši ciljanu imputacija -  svuda gde podatak o vlažnosti nedostaje, a raspolažemo temperaturom i tačkom rose, formula automatski popunjava praznine egzaktnim vrednostima, a potom generiše  finalni izveštaj o broju uspešno popunjenih redova i preostalim prazninama.

In [ ]:
def izracunaj_vlaznost(temp, tacka_rose):
    e = np.exp((17.625 * tacka_rose) / (243.04 + tacka_rose))
    e_s = np.exp((17.625 * temp) / (243.04 + temp))
    rh = 100 * (e / e_s)
    return np.clip(rh, 0, 100)

def procesuiraj_vreme(df, vreme):
    temp_kol = f'Temp{vreme}'
    rose_kol = f'TackaRose{vreme}'
    hum_kol = f'Humidity{vreme}'

    print(f"\n{'='*40}")
    print(f" Izveštaj za termin: {vreme.upper()}")
    print(f"{'='*40}")
    df[temp_kol] = pd.to_numeric(df[temp_kol], errors='coerce')
    df[rose_kol] = pd.to_numeric(df[rose_kol], errors='coerce')
    df[hum_kol] = pd.to_numeric(df[hum_kol], errors='coerce')
    imputacija_maska = df[hum_kol].isna() & df[temp_kol].notna() & df[rose_kol].notna()
    broj_za_popunu = imputacija_maska.sum()

    print(f"[Imputacija] Identifikovano za popunjavanje: {broj_za_popunu} redova")

    if broj_za_popunu > 0:
        df.loc[imputacija_maska, hum_kol] = izracunaj_vlaznost(
            df.loc[imputacija_maska, temp_kol],
            df.loc[imputacija_maska, rose_kol]
        )
        print(" -> Popunjavanje uspešno izvršeno!")

    preostalo_maska = df[hum_kol].isna()
    broj_preostalih = preostalo_maska.sum()

    preostalo_ima_rose = df.loc[preostalo_maska, rose_kol].notna().sum()

    print(f"\n[Izveštaj] Neuspešno popunjeno (konačan broj nedostajućih): {broj_preostalih} redova")
    if broj_preostalih > 0:
        print(f" -> Od tih {broj_preostalih}, podatak o tački rose postoji u {preostalo_ima_rose} redova.")
        if preostalo_ima_rose > 0:
            print("    (Ovi redovi nisu popunjeni jer nedostaje podatak o trenutnoj temperaturi)")
    return df

data = procesuiraj_vreme(data, '9am')
data = procesuiraj_vreme(data, '3pm')

Za redove koji nisu uspesno popunjeni prethodnom metodom, upotrebicemo random forest.

In [ ]:
data = imputeData("Humidity9am",data, 
                          ['Temp3pm', 'Temp9am',"Temp9am_Min_diff","Max_Temp3pm_diff",
                           "Max_Min_Temp_Diff",  'Rainfall', 'WindSpeed9am', 
                           "Humidity3pm", "Lon"], True, None)

In [ ]:
data = imputeData("Humidity3pm",data, 
                          ["Humidity9am", "Pressure3pm", "Temp3pm", "Max_Min_Temp_Diff", 
                           "Temp3pm_rolling_mean_3", "Temp9am_Min_diff", "Max_Temp3pm_diff",
                            "Pressure3pm_pre_1_dana","Inv_Dist_Jug"], 
                           True,"WeatherAus_After_11_2_3_7.csv")

##### 11.2.3.8 Sunshine

Atribut Sunshine je nakon prostorne imputacije po grupama stanica (odeljak 4.1) i dalje imao oko 44.6% nedostajucih vrednosti, jer prostorni pristup ne može popuniti podatke kada i susedne stanice u datom trenutku nemaju merenje. Buduci da je Sunshine, i pored visokog stepena nedostajanja, jedan od najznacajnijih prediktora za RainTomorrow (odeljak 3), sprovedena je ablation studija koja je na identicnom hronološkom test skupu uporedila pet strategija: potpuno izbacivanje atributa, globalnu medijanu, medijanu po lokaciji i mesecu, IterativeImputer, i Random Forest imputaciju (istom metodologijom kao u ovom odeljku). Random Forest imputacija bez dodatnog indikatora dala je najbolji F1 i ROC-AUC, dok je verzija sa Sunshine_missing indikatorom dala najbolju tacnost/preciznost. Nasuprot tome, strategija koriscena u ranijoj verziji ovog rada (treniranje dva odvojena modela - jedan za redove sa Sunshine podatkom, drugi bez) pokazala se kao najslabija opcija na sve tri metrike. Zbog toga se Sunshine ovde popunjava istom Random Forest metodom kao Humidity, cime se izbegava potreba za deljenjem skupa u fazi modelovanja.

In [ ]:
data = imputeData("Sunshine", data,
                          ['Cloud9am', 'Cloud3pm', 'Humidity9am', 'Humidity3pm', 
                           'Rainfall','MaxTemp', 'Temp3pm', 'Pressure9am', 'Pressure3pm',
                           'Max_Min_Temp_Diff', 'DayOfYear_Sin', 'DayOfYear_Cos', 
                           'WindGustSpeed'], True, "WeatherAus_After_11_2_3_8.csv")

#### 11.2.4 Provera kvaliteta imputacije: R² i vizuelno poređenje sa prostom medijanom

Kako bismo potvrdili da nasa kompleksna (prostorno-grupna + Random Forest) imputacija zaista cuva karakteristike originalne raspodele podataka, u nastavku prvo kvantitativno poredimo kvalitet imputacije (R², 5-fold unakrsna validacija na poznatim vrednostima, isti princip kao u `imputeData()`) izmedju naseg Random Forest pristupa i proste globalne medijane. Zatim vizuelno poredimo raspodele (histogram/KDE) svih imputiranih kolona - original (samo poznate vrednosti) naspram privremene medijana-imputacije naspram nase stvarne imputacije - kao i detaljniji prikaz po lokaciji za `Sunshine` (kolona sa najvecim procentom nedostajucih vrednosti).

In [ ]:
# R^2 poredjenje: RF (nasa metoda) vs prosta medijana, 5-fold CV na poznatim vrednostima
_kolone_za_proveru = ["MinTemp", "MaxTemp", "Temp9am", "Temp3pm", "Pressure9am", "Pressure3pm",
                       "Humidity9am", "Humidity3pm", "Sunshine"]

_prediktori_za_proveru = {
    "MinTemp": ['Temp3pm_rolling_mean_3', 'Pressure9am', 'Temp9am', 'Temp3pm', 'Year_Scaled', 'DayOfYear_Sin', 'DayOfYear_Cos', 'WindSpeed9am', 'Humidity9am'],
    "MaxTemp": ['Temp3pm_rolling_mean_3', 'Pressure3pm', 'Temp9am', 'Temp3pm', 'Year_Scaled', 'DayOfYear_Sin', 'DayOfYear_Cos', 'WindSpeed3pm', 'Humidity3pm'],
    "Temp9am": ['Temp3pm_rolling_mean_3', 'Pressure3pm', 'MinTemp', 'Temp3pm', 'Year_Scaled', 'DayOfYear_Sin', 'DayOfYear_Cos', 'WindSpeed3pm', 'Humidity3pm'],
    "Temp3pm": ['Temp3pm_rolling_mean_3', 'Pressure3pm', 'MinTemp', 'Temp9am', 'Year_Scaled', 'DayOfYear_Sin', 'DayOfYear_Cos', 'WindSpeed3pm', 'Humidity3pm'],
    "Pressure9am": ["Pressure3pm", 'Temp9am', 'Temp3pm', 'Year_Scaled', 'DayOfYear_Sin', 'DayOfYear_Cos', 'Pressure9am_pre_1_dana', 'Pressure3pm_pre_1_dana', 'Temp9am_Min_diff', 'Max_Temp3pm_diff', 'WindSpeed3pm', 'WindDir3pm_sin', 'WindDir3pm_cos', 'WindDir9am_sin', 'WindDir9am_cos', 'WindSpeed3pm_rolling_mean_3'],
    "Pressure3pm": ['Pressure9am', 'Temp9am', 'Temp3pm', 'Year_Scaled', 'DayOfYear_Sin', 'DayOfYear_Cos', 'Pressure9am_pre_1_dana', 'Pressure3pm_pre_1_dana', 'Temp9am_Min_diff', 'Max_Temp3pm_diff', 'WindSpeed3pm', 'WindDir3pm_sin', 'WindDir3pm_cos', 'WindDir9am_sin', 'WindDir9am_cos', 'WindSpeed3pm_rolling_mean_3'],
    "Humidity3pm": ["Humidity9am", "Pressure3pm", "Temp3pm", "Max_Min_Temp_Diff", "Temp3pm_rolling_mean_3", "Temp9am_Min_diff", "Max_Temp3pm_diff", "Pressure3pm_pre_1_dana", "Inv_Dist_Jug"],
    "Humidity9am": ['Temp3pm', 'Temp9am', "Temp9am_Min_diff", "Max_Temp3pm_diff", "Max_Min_Temp_Diff", 'Rainfall', 'WindSpeed9am', "Humidity3pm", "Lon"],
    "Sunshine": ['Cloud9am', 'Cloud3pm', 'Humidity9am', 'Humidity3pm', 'Rainfall', 'MaxTemp', 'Temp3pm', 'Pressure9am', 'Pressure3pm', 'Max_Min_Temp_Diff', 'DayOfYear_Sin', 'DayOfYear_Cos', 'WindGustSpeed'],
}


def _r2_poredjenje(df, is_train_mask, kolona, prediktori):
    prediktori_full = list(dict.fromkeys(prediktori + core_features))
    poznati = df[df[kolona].notna() & is_train_mask].copy()
    imputer_tmp = SimpleImputer(strategy='median')
    X = imputer_tmp.fit_transform(poznati[prediktori_full])
    y = poznati[kolona].values

    rf_r2, med_r2 = [], []
    for i in range(5):
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.1, random_state=i)
        rf_tmp = RandomForestRegressor(n_estimators=100, random_state=i, n_jobs=-1)
        rf_tmp.fit(X_tr, y_tr)
        rf_r2.append(r2_score(y_te, rf_tmp.predict(X_te)))
        med_pred = np.full_like(y_te, fill_value=np.median(y_tr), dtype=float)
        med_r2.append(r2_score(y_te, med_pred))
    return np.mean(rf_r2), np.mean(med_r2)

data_pre_imputacije = loadData("backups/WatherAus_After_11_2_1.csv")
_is_train_pre = data_pre_imputacije['Date'] < SPLIT_DATE
_r2_redovi = []
for _kol in _kolone_za_proveru:
    _rf_r2, _med_r2 = _r2_poredjenje(data_pre_imputacije, _is_train_pre, _kol, _prediktori_za_proveru[_kol])
    _r2_redovi.append({"kolona": _kol, "RF_R2": _rf_r2, "Medijana_R2": _med_r2, "razlika": _rf_r2 - _med_r2})

r2_poredjenje_tabela = pd.DataFrame(_r2_redovi)
print("R2 poredjenje: nasa (RF) imputacija naspram proste medijane (5-fold CV, na poznatim vrednostima)")
print(r2_poredjenje_tabela.round(4).to_string(index=False))


In [ ]:
# Privremena medijana-imputacija (samo za vizuelno poredjenje ispod - NE koristi se
# dalje u analizi niti utice na stvarni 'data')
data_medijana_privremeno = data_pre_imputacije.copy()
for _kol in _kolone_za_proveru:
    _med = data_medijana_privremeno.loc[_is_train_pre, _kol].median()
    data_medijana_privremeno[_kol] = data_medijana_privremeno[_kol].fillna(_med)

In [ ]:
# Histogrami: original (poznate vrednosti) vs privremena medijana vs nasa (RF/prostorna) imputacija
fig, axes = plt.subplots(3, 3, figsize=(16, 13))
axes = axes.flatten()
for ax, _kol in zip(axes, _kolone_za_proveru):
    _orig = data_pre_imputacije.loc[data_pre_imputacije[_kol].notna(), _kol]
    sns.kdeplot(_orig, ax=ax, label="Original (poznate vrednosti)", color="black", linewidth=2)
    sns.kdeplot(data_medijana_privremeno[_kol], ax=ax, label="Medijana (privremeno)", color="red", linestyle="--")
    sns.kdeplot(data[_kol], ax=ax, label="Nasa (RF/prostorna)", color="teal")
    ax.set_title(_kol)
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
# Sunshine po lokaciji - medijana vs nasa (RF/prostorna) imputacija (flagship primer,
# najveci % nedostajucih vrednosti, ukljucujuci lokacije sa 100% missing pre imputacije)
_lokacije = sorted(data_pre_imputacije["Location"].unique())
_n = len(_lokacije)
_cols_n = 7
_rows_n = -(-_n // _cols_n)

for _variant_name, _variant_df, _color in [("medijana (privremeno)", data_medijana_privremeno, "red"),
                                            ("nasa (RF/prostorna)", data, "teal")]:
    fig, axes = plt.subplots(_rows_n, _cols_n, figsize=(_cols_n * 3, _rows_n * 2.2), sharex=True)
    axes = axes.flatten()
    for ax, _lok in zip(axes, _lokacije):
        _podaci = _variant_df.loc[_variant_df["Location"] == _lok, "Sunshine"].dropna()
        sns.histplot(_podaci, ax=ax, bins=20, color=_color)
        ax.set_title(_lok, fontsize=8)
        ax.set_xlabel("")
        ax.set_ylabel("")
    for ax in axes[_n:]:
        ax.axis("off")
    fig.suptitle(f"Distribucija Sunshine po lokaciji - {_variant_name}", fontsize=14)
    plt.tight_layout()
    plt.show()

Kao sto se vidi iz R² tabele, prosta medijana ima R² blizu 0 (ocekivano - konstantna predikcija po definiciji ne objasnjava nikakvu varijansu), dok nasa Random Forest imputacija postize R² od 0.79 (Sunshine, najteza kolona) do preko 0.98 (MaxTemp). Histogrami potvrdjuju isto vizuelno: kriva nase imputacije se gotovo preklapa sa originalnom (poznatom) raspodelom u svih 9 kolona, dok medijana pravi izrazit vestacki "siljak" tacno na medijani - narocito drasticno kod `Sunshine`, gde lokacije koje su ranije imale 100% nedostajucih vrednosti (npr. `Albury`, `BadgerysCreek`, `Ballarat`, `Katherine`) kod medijane dobijaju jedan besmislen spike umesto ikakve realne raspodele, dok nasa prostorna imputacija daje uverljivu, po lokaciji specificnu raspodelu.

In [ ]:
data = kreirajAnalizu(data)